# Ingest plan — programmatic fetch (StatCan WDS + CMHC)

This notebook searches StatCan's Web Data Service (WDS) for relevant product IDs (PIDs) and scrapes CMHC table pages for CSV links, then downloads matching CSVs to `data/raw/`. Run cells in order.

In [1]:
# Cell 1: setup and imports
import requests
import re
import json
import time
from pathlib import Path
import pandas as pd

PROJECT = Path('.')
RAW = PROJECT / 'data' / 'raw'
PROCESSED = PROJECT / 'data' / 'processed'
RAW.mkdir(parents=True, exist_ok=True)
PROCESSED.mkdir(parents=True, exist_ok=True)

# StatCan WDS base endpoints
STATCAN_WDS_BASE = 'https://www150.statcan.gc.ca/t1/wds/rest'
ALL_CUBES_LITE = f'{STATCAN_WDS_BASE}/getAllCubesListLite/en'
FULL_TABLE_CSV = f'{STATCAN_WDS_BASE}/getFullTableDownloadCSV'  # append /{PID}/en

print('setup complete')

ModuleNotFoundError: No module named 'requests'

In [ ]:
# Cell 2: helpers for StatCan WDS
_statcan_cubes_cache = None

def fetch_all_statcan_cubes(force=False):
    global _statcan_cubes_cache
    if _statcan_cubes_cache is not None and not force:
        return _statcan_cubes_cache
    resp = requests.get(ALL_CUBES_LITE, timeout=30)
    resp.raise_for_status()
    data = resp.json()
    # data expected as list of cube metadata objects with 'title' and 'pid'
    _statcan_cubes_cache = data
    return data

def statcan_search(keyword, max_results=10):
    keyword = keyword.lower()
    cubes = fetch_all_statcan_cubes()
    matches = []
    for c in cubes:
        title = (c.get('title') or c.get('productTitle') or '')
        pid = c.get('pid') or c.get('productId') or c.get('productID')
        if not title or not pid:
            continue
        if keyword in title.lower():
            matches.append({'pid': pid, 'title': title})
        if len(matches) >= max_results:
            break
    return matches

def download_statcan_table_csv(pid, dest_path):
    url = f'{FULL_TABLE_CSV}/{pid}/en'
    dest = Path(dest_path)
    if dest.exists():
        print(dest, 'exists')
        return dest
    print('downloading', url)
    resp = requests.get(url, stream=True, timeout=60)
    resp.raise_for_status()
    with open(dest, 'wb') as f:
        for chunk in resp.iter_content(1024*1024):
            if chunk:
                f.write(chunk)
    print('wrote', dest)
    return dest

In [ ]:
# Cell 3: helpers for CMHC page scraping and generic downloading
def find_csv_links_on_page(page_url):
    try:
        r = requests.get(page_url, timeout=30)
        r.raise_for_status()
    except Exception as e:
        print('failed to fetch', page_url, e)
        return []
    html = r.text
    # find href="..." values
    hrefs = re.findall(r'href=[\"\]([^\"\]+)[\"\]', html, flags=re.I)
    candidates = []
    for h in hrefs:
        if h.lower().endswith('.csv') or h.lower().endswith('.xlsx') or ('download' in h.lower() and ('.csv' in h.lower() or '.xlsx' in h.lower())):
            # make absolute if needed
            if h.startswith('//'):
                h = 'https:' + h
            elif h.startswith('/'):
                base = re.match(r'(https?://[^/]+)', page_url)
                if base:
                    h = base.group(1) + h
            candidates.append(h)
    # dedupe
    seen = []
    out = []
    for c in candidates:
        if c not in seen:
            seen.append(c)
            out.append(c)
    return out

def download_file(url, dest_path):
    dest = Path(dest_path)
    if dest.exists():
        print(dest, 'exists')
        return dest
    print('downloading', url)
    r = requests.get(url, stream=True, timeout=60)
    r.raise_for_status()
    with open(dest, 'wb') as f:
        for chunk in r.iter_content(1024*1024):
            if chunk:
                f.write(chunk)
    print('wrote', dest)
    return dest

In [ ]:
# Cell 4: dataset definitions (keywords / CMHC pages)
datasets = [
    { 'name': 'median_household_income', 'provider': 'statcan', 'keyword': 'median total income' },
    { 'name': 'population_estimates', 'provider': 'statcan', 'keyword': 'population estimates cma' },
    { 'name': 'unemployment_rate', 'provider': 'statcan', 'keyword': 'unemployment rate cma' },
    { 'name': 'cpi_all_items', 'provider': 'statcan', 'keyword': 'consumer price index all-items' },
    { 'name': 'rental_market_rents', 'provider': 'cmhc', 'page': 'https://www.cmhc-schl.gc.ca/professionals/housing-markets-data-and-research/housing-data/data-tables/rental-market' },
    { 'name': 'housing_starts', 'provider': 'cmhc', 'page': 'https://www.cmhc-schl.gc.ca/professionals/housing-markets-data-and-research/housing-data/data-tables/housing-market-data' }
]

print('datasets defined')

In [ ]:
# Cell 5: run fetch for each dataset (statcan via WDS, cmhc via page scrape)
for ds in datasets:
    name = ds['name']
    try:
        if ds['provider'] == 'statcan':
            print('
Searching StatCan for', name, 'keyword=', ds.get('keyword'))
            matches = statcan_search(ds.get('keyword', ''), max_results=10)
            if not matches:
                print('no matches for', name)
                continue
            # show top candidates
            for i,m in enumerate(matches[:5],1):
                print(i, m['pid'], m['title'])
            pid = matches[0]['pid']
            dest = RAW / f"{name}.csv"
            try:
                download_statcan_table_csv(pid, dest)
            except Exception as e:
                print('statcan download failed', e)
            time.sleep(1)
        elif ds['provider'] == 'cmhc':
            page = ds.get('page')
            print('
Searching CMHC page for', name, page)
            links = find_csv_links_on_page(page)
            if not links:
                print('no csv links found on', page)
                continue
            for link in links[:3]:
                print('candidate:', link)
            # download first candidate
            dest = RAW / f"{name}.{links[0].split('.')[-1]}"
            try:
                download_file(links[0], dest)
            except Exception as e:
                print('download failed', e)
            time.sleep(1)
    except Exception as e:
        print('error processing', name, e)

In [ ]:
# Cell 6: quick QA of downloaded raw files
import glob
for p in glob.glob(str(RAW / '*')):
    try:
        size = Path(p).stat().st_size
        print(p, 'size=', size)
    except Exception as e:
        print('stat failed', p, e)